# Trabalho 1 — Aquisição de Dados

## 1. Identificação do projeto

**Tema:**

Análise da relação entre a variação dos preços da cesta básica, a inflação dos alimentos e os diferentes períodos de governos presidenciais e estaduais no Brasil desde 1994.

**Integrantes:**

- Nome 1
- Nome 2
- Nome 3
- Nome 4

**Disciplina:** Ciência de Dados

**Instituição:** Universidade Federal do Amazonas — Instituto de Computação

## 2. Pergunta motivadora

De que maneira a variação nos preços da cesta básica e a inflação dos alimentos se
comportaram ao longo das diferentes gestões presidenciais e estaduais no Brasil
desde 1994, e como os ciclos eleitorais e espectros partidários se correlacionam com
esses momentos de instabilidade?

## 3. Objetivo da coleta

Construir uma base de dados integrada contendo informações sobre
a inflação dos alimentos, os preços da cesta básica e informações
relacionadas aos períodos de governos e eleições no Brasil.

A base será utilizada nas etapas posteriores do projeto para
investigar possíveis padrões entre as variáveis econômicas e
políticas.

## 4. Bibliotecas e configurações

### 4.1 Bibliotecas utilizadas

| Biblioteca | Finalidade | Instalação |
|---|---|---|
| `requests` | Requisições HTTP para a API do IBGE/SIDRA e tratamento de status HTTP | `pip install requests` |
| `pandas` | Leitura, limpeza, transformação e integração de dados; leitura de HTML com `read_html` | `pip install pandas` |
| `beautifulsoup4` | Parsing e extração de dados do HTML da página do DIEESE | `pip install beautifulsoup4` |
| `lxml` | Parser HTML/XML de alta performance (motor alternativo para BeautifulSoup) | `pip install lxml` |
| `zipfile` (padrão) | Extração de arquivos CSV de arquivos ZIP baixados do TSE (CDN) | - |
| `io` (padrão) | Manipulação de bytes em memória durante o download/decompresão | - |
| `pyarrow` | Leitura e escrita de arquivos Parquet (formato recomendado para a base tratada) | `pip install pyarrow` |
| `json` (padrão) | Processamento de respostas JSON da API | - |
| `datetime` (padrão) | Registro de data e hora das coletas com fuso horário | - |
| `time` (padrão) | Pausas entre requisições para respeitar limites dos servidores | - |
| `pathlib` (padrão) | Construção de caminhos de arquivos de forma portável | - |

### 4.2 Configurações iniciais

- **User-Agent:** identificar o robô em requisições HTTP, conforme recomendado em `robots.txt`.
- **Timeout:** definir limite de espera por requisição (ex: 30 segundos).
- **Pausas entre requisições:** intervalo de pelo menos 1 segundo entre chamadas à API e entre páginas scrapeadas.
- **Codificação de resposta:** forçar `utf-8` ao ler HTML ou JSON para evitar problemas de acentuação.
- **Seed de data/hora:** registrar cada coleta com `datetime.now(timezone.utc)` ou fuso horário de Brasilia (`America/Sao_Paulo`).


In [ ]:
# Bibliotecas e configurações
import requests
import pandas as pd
import json
import os
import io
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import time

# Parser HTML
from bs4 import BeautifulSoup

# Exportacao Parquet
import pyarrow as pa
import pyarrow.parquet as pq

# Configuracoes gerais
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Fuso horario de referencia (Brasilia)
FUSO = "America/Sao_Paulo"

# Caminhos base (ajuste conforme a estrutura da sua maquina)
BASE_DIR = Path.cwd()
BRUTOS_DIR = BASE_DIR / "projeto" / "dados_brutos"
TRATADOS_DIR = BASE_DIR / "projeto" / "dados_tratados"
DOC_DIR = BASE_DIR / "projeto" / "documentacao"

# Criar pastas se nao existirem
BRUTOS_DIR.mkdir(parents=True, exist_ok=True)
TRATADOS_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

# Configuracoes de requisicao
HEADERS = {
    "User-Agent": "TrabalhoAcademico-CienciaDeDados/1.0 (contato@exemplo.com)"
}
TIMEOUT = 30  # segundos
PAUSA = 1.5   # segundos entre requisicoes

print("Bibliotecas carregadas e pastas configuradas.")


## 5. Fonte 1 — IBGE/SIDRA (API)

### 5.1 Descrição da fonte

O Sistema IBGE de Recuperação Automática (SIDRA) é uma plataforma do Instituto Brasileiro de Geografia e Estatística (IBGE) que disponibiliza dados estatísticos oficiais do Brasil. Para este trabalho, será utilizada a API do SIDRA para adquirir dados relacionados ao Índice Nacional de Preços ao Consumidor Amplo (IPCA), com foco nos preços e na inflação dos alimentos. Esses dados serão utilizados para analisar a evolução dos preços ao longo do período estudado e posteriormente integrados aos dados obtidos por Web Scraping.


### 5.2 Definição dos dados

Será utilizada a **Tabela 61 do Sistema Nacional de Índices de Preços ao Consumidor (SNIPC)**, disponibilizada pelo IBGE por meio do SIDRA:

- **Indicador:** Índice Nacional de Preços ao Consumidor Amplo (IPCA);
- **Tabela:** 61 — *IPCA — Peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços*;
- **Período disponível na tabela:** janeiro de 1991 a julho de 1999;
- **Dimensão temporal:** mês e ano de referência;
- **Dimensão dos produtos e serviços:** índice geral, grupos, subgrupos, itens e subitens;
- **Variável principal:** peso mensal de cada grupo, subgrupo, item ou subitem na composição do IPCA;
- **Unidade de observação:** uma categoria de produto ou serviço observada em determinado mês e ano;
- **Finalidade na pesquisa:** representar a participação relativa dos alimentos e de outros grupos de consumo na composição do IPCA, permitindo comparar a evolução da inflação dos alimentos com os períodos de governos e ciclos eleitorais.

A tabela será utilizada como fonte de informações sobre a estrutura mensal do IPCA no período inicial da série histórica analisada. Para a comparação com os preços da cesta básica, será necessário selecionar as categorias relacionadas à alimentação e documentar os códigos, níveis e classificações escolhidos no SIDRA.

**Observação sobre a cobertura temporal:** a Tabela 61 termina em julho de 1999. Como o projeto pretende analisar o período desde 1994 e pode exigir dados posteriores a 1999, será necessário verificar, em etapa posterior, se outras tabelas do SIDRA devem ser incorporadas para complementar a série do IPCA e manter a cobertura temporal do estudo.

**Fonte da definição:** [Tabela 61 — IPCA no SIDRA](https://sidra.ibge.gov.br/Tabela/61), incluindo as [notas da tabela](https://sidra.ibge.gov.br/Tabela/61#notas-tabela).

### 5.3 URL da API

[https://sidra.ibge.gov.br/Tabela/61](https://sidra.ibge.gov.br/Tabela/61)



In [ ]:
### 5.3 Teste da requisição
import requests

url = "https://sidra.ibge.gov.br/Tabela/61"

resposta = requests.get(url)

print(resposta.status_code)
print(resposta.text)

if resposta.status_code == 200:
    print("Requisição realizada com sucesso.")
else:
    print("Erro na requisição:", resposta.status_code)

In [ ]:
# 5.5 Coleta completa
# Preencher posteriormente.

In [ ]:
# 5.6 Salvamento dos dados brutos
# Preencher posteriormente.

## 6. Fonte 2 — DIEESE (Web Scraping)

### 6.1 Descrição da fonte

*Preencher posteriormente.*

### 6.2 URL

*Preencher posteriormente.*

### 6.3 Estrutura HTML

*Preencher posteriormente.*

In [ ]:
# 6.4 Scraping
# Preencher posteriormente.

In [ ]:
# 6.5 Salvamento dos dados brutos
# Preencher posteriormente.

## 7. Fonte 3 — TSE/Dados Abertos (API)

### 7.1 Descrição da fonte

O Portal de Dados Abertos do Tribunal Superior Eleitoral (TSE) é uma plataforma que disponibiliza dados eleitorais oficiais do Brasil. Para este trabalho, serão utilizadas duas APIs do TSE:

1. **API CKAN** — para consulta de metadados e obtenção de URLs de recursos (datasets):
   `https://dadosabertos.tse.jus.br/api/3/action`
   - Endpoint `package_search`: busca datasets por termo (ex: `candidatos`, `resultados`).
   - Endpoint `package_show`: retorna metadados completos de um dataset, incluindo os recursos (arquivos) disponíveis.
   - Endpoint `resource_show`: retorna detalhes de um recurso específico, incluindo a URL de download.
   - Todos os recursos retornam dados no formato JSON, seguindo o padrão CKAN.

2. **API DivulgaCandContas (REST)** — para acesso em tempo real a informações sobre eleições, candidatos e suas respectivas informações:
   `https://divulgacandcontas.tse.jus.br/divulga/rest/v1`
   - Endpoint `/eleicao/ordinarias`: lista todas as eleições ordinárias disponíveis, com IDs, anos, tipos e datas.
   - Endpoint `/candidatura/listar/{ano}/{municipio_cod}/{eleicao_id}/{cargo_cod}/candidatos`: lista candidatos de um cargo específico em uma eleição.
   - Endpoint `/candidatura/buscar/{ano}/{municipio_cod}/{eleicao_id}/candidato/{candidato_id}`: retorna detalhes completos de um candidato.

Para o tema deste projeto, os dados relevantes são:

- **Datasets de Candidatos** (`candidatos-{ano}`): contêm informações sobre candidatos, incluindo nome, partido (sigla), cargo, ano e UF. Disponíveis para os anos: 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
- **Datasets de Resultados** (`resultados-{ano}`): contêm resultados eleitorais detalhados por município e zona. Disponíveis a partir de 1994.

Esses dados permitem mapear:

- os governos presidenciais e estaduais ao longo do tempo (períodos de 1994 a 2022);
- os ciclos eleitorais (anos de eleições gerais e municipais);
- os espectros partidários dos governantes (siglas de partidos e coligações).

Como os dados são disponibilizados em arquivos ZIP contendo CSVs, a coleta envolve a API CKAN para obter os metadados e URLs, seguida do download e descompressão dos arquivos.

**Fonte da definição:** [Portal de Dados Abertos do TSE](https://dadosabertos.tse.jus.br/) e [DivulgaCandContas REST API](https://divulgacandcontas.tse.jus.br/divulga/rest/v1).

### 7.2 URL da API

**API CKAN (Datasets):**

`https://dadosabertos.tse.jus.br/api/3/action`

**API DivulgaCandContas (Tempo real):**

`https://divulgacandcontas.tse.jus.br/divulga/rest/v1`

### 7.3 Endpoints e parâmetros

| Endpoint | Método | Parâmetros | Descrição |
|---|---|---|---|
| `package_search` | GET | `q=<termo>`, `rows=<n>` | Busca datasets por termo |
| `package_show` | GET | `id=<slug>` | Metadados e recursos de um dataset |
| `resource_show` | GET | `id=<resource_id>` | URL e metadados de um recurso |
| `/eleicao/ordinarias` | GET | — | Lista eleições ordinárias |
| `/candidatura/listar/{ano}/{municipio_cod}/{eleicao_id}/{cargo_cod}/candidatos` | GET | — | Lista candidatos de um cargo |

### 7.4 Dados para este projeto

- **Cargo Presidencial (código 1)**: anos eleitorais 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
- **Cargo Governador (código 3)**: mesmos anos eleitorais.
- **Chave de integração**: `ano` (ano eleitoral) e `sigla_partido` (partido do candidato).
- **Licença**: Creative Commons Atribuição (CC-BY).



In [ ]:
### 7.5 Teste da requisição
import requests

# API CKAN - buscar datasets de candidatos
CKAN_URL = "https://dadosabertos.tse.jus.br/api/3/action"
DIVULGA_URL = "https://divulgacandcontas.tse.jus.br/divulga/rest/v1"

# Teste 1: package_search por 'candidatos'
r = requests.get(
    f"{CKAN_URL}/package_search",
    params={"q": "candidatos", "rows": 20},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"package_search candidatos: HTTP {r.status_code}")

# Teste 2: package_show para 'candidatos-1994'
r2 = requests.get(
    f"{CKAN_URL}/package_show",
    params={"id": "candidatos-1994"},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"package_show candidatos-1994: HTTP {r2.status_code}")
if r2.status_code == 200:
    ds = r2.json()["result"]
    print(f"  Título: {ds.get('title')}")
    print(f"  Licença: {ds.get('license_id')}")
    print(f"  Recursos: {len(ds.get('resources', []))}")
    for res in ds.get("resources", [])[:3]:
        print(f"    - {res.get('name')}: {res.get('url')[:80]}...")

# Teste 3: DivulgaCandContas - eleicoes ordinarias
r3 = requests.get(
    f"{DIVULGA_URL}/eleicao/ordinarias",
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"\neleicao/ordinarias: HTTP {r3.status_code}")
if r3.status_code == 200:
    eleicoes = r3.json()
    print(f"  Eleições encontradas: {len(eleicoes)}")
    for e in eleicoes[:5]:
        print(f"    - {e.get('ano')}: {e.get('nomeEleicao')} (ID: {e.get('id')}, tipo: {e.get('tipoAbrangencia')})")

# Teste 4: package_search por 'resultados'
r4 = requests.get(
    f"{CKAN_URL}/package_search",
    params={"q": "resultados", "rows": 20},
    headers=HEADERS,
    timeout=TIMEOUT,
)
print(f"\npackage_search resultados: HTTP {r4.status_code}")
if r4.status_code == 200:
    results = r4.json()["result"]["results"]
    for item in results[:8]:
        print(f"  - {item.get('name')}")


In [ ]:
# 7.6 Coleta completa
#
# Anos eleitorais de interesse (presidenciais e gubernatoriais desde 1994):
# 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022.
#
# Estratégia:
# 1. Consultar a API CKAN para obter os recursos (resources) de download (ZIPs)
#    dos datasets 'candidatos-{ano}' e 'resultados-{ano}'.
# 2. Baixar os arquivos ZIP brutos conservando-os em dados_brutos/tse/.
# 3. Registrar a data/hora da coleta, status HTTP e metadados.

# Anos de eleicoes gerais (presidenciais e gubernatoriais)
ANOS_ELEICAO = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

# Pasta para arquivos brutos do TSE
TSE_BRUTOS_DIR = BRUTOS_DIR / "tse"
TSE_BRUTOS_DIR.mkdir(parents=True, exist_ok=True)

# Data/hora da coleta (UTC)
data_hora_coleta = datetime.now(timezone.utc)
print(f"Coleta iniciada em: {data_hora_coleta.isoformat()}")

# Lista para registrar a proveniencia de cada recurso
proveniencia_tse = []

# Função para obter recursos (resources) de um dataset via API CKAN
def obter_recursos_dataset(slug: str) -> list:
    """Usa a API CKAN para obter a lista de recursos de um dataset."""
    try:
        r = requests.get(
            f"{CKAN_URL}/package_show",
            params={"id": slug},
            headers=HEADERS,
            timeout=TIMEOUT,
        )
        if r.status_code == 200:
            return r.json().get("result", {}).get("resources", [])
        print(f"  Erro HTTP {r.status_code} para dataset '{slug}'")
    except Exception as e:
        print(f"  Erro de conexão para '{slug}': {e}")
    return []

# Função para baixar um arquivo e salvar o bruto
def baixar_arquivo(url: str, destino: Path) -> dict:
    """Baixa um arquivo da URL e salva em destino. Retorna dict com proveniencia."""
    info = {
        "url": url,
        "status_http": None,
        "arquivo": str(destino),
        "tamanho_bytes": None,
    }
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        info["status_http"] = r.status_code
        if r.status_code == 200:
            destino.parent.mkdir(parents=True, exist_ok=True)
            with open(destino, "wb") as f:
                f.write(r.content)
            info["tamanho_bytes"] = len(r.content)
        else:
            print(f"  Erro HTTP {r.status_code} ao baixar: {url[:80]}...")
    except Exception as e:
        info["status_http"] = "erro"
        print(f"  Erro de conexão: {e}")
    return info

# Coletar datasets de candidatos e resultados para cada ano
for ano in ANOS_ELEICAO:
    print(f"\nProcessando ano: {ano}")

    # --- Candidatos ---
    slug_cand = f"candidatos-{ano}"
    recursos_cand = obter_recursos_dataset(slug_cand)
    for rec in recursos_cand:
        if rec.get("format", "").upper() == "CSV":
            nome_arquivo = f"{rec['name'].replace(' ', '_')}_{ano}.zip"
            destino = TSE_BRUTOS_DIR / nome_arquivo
            info = baixar_arquivo(rec["url"], destino)
            proveniencia_tse.append({
                "fonte_id": "TSE-CKAN",
                "dataset": slug_cand,
                "recurso": rec["name"],
                "data_hora_coleta": data_hora_coleta.isoformat(),
                **info,
            })
            time.sleep(PAUSA)

    # --- Resultados (se disponível) ---
    slug_res = f"resultados-{ano}"
    recursos_res = obter_recursos_dataset(slug_res)
    for rec in recursos_res:
        if rec.get("format", "").upper() == "CSV":
            nome_arquivo = f"{rec['name'].replace(' ', '_')}_{ano}.zip"
            destino = TSE_BRUTOS_DIR / nome_arquivo
            info = baixar_arquivo(rec["url"], destino)
            proveniencia_tse.append({
                "fonte_id": "TSE-CKAN",
                "dataset": slug_res,
                "recurso": rec["name"],
                "data_hora_coleta": data_hora_coleta.isoformat(),
                **info,
            })
            time.sleep(PAUSA)

print(f"\nColeta concluída. Recursos baixados: {len(proveniencia_tse)}")
for p in proveniencia_tse:
    print(f"  {p['dataset']}/{p['recurso']}: HTTP {p['status_http']}")


In [ ]:
# 7.7 Salvamento dos dados brutos
#
# Os arquivos ZIP ja foram salvos em dados_brutos/tse/ durante a coleta.
# Aqui salvamos tambem a resposta bruta da API CKAN (JSON) e o registro
# de proveniencia consolidado.

# Salvar resposta bruta da API CKAN (package_show para um dataset exemplo)
ds_exemplo = requests.get(
    f"{CKAN_URL}/package_show",
    params={"id": "candidatos-2022"},
    headers=HEADERS,
    timeout=TIMEOUT,
)
raw_api_file = BRUTOS_DIR / f"tse_ckan_api_resposta_{data_hora_coleta.strftime('%Y%m%d_%H%M%S')}.json"
with open(raw_api_file, "w", encoding="utf-8") as f:
    json.dump(ds_exemplo.json(), f, ensure_ascii=False, indent=2)
print(f"Resposta bruta da API CKAN salva em: {raw_api_file}")

# Salvar registro de proveniencia do TSE
proveniencia_df = pd.DataFrame(proveniencia_tse)
proveniencia_file = BRUTOS_DIR / f"proveniencia_tse_{data_hora_coleta.strftime('%Y%m%d_%H%M%S')}.csv"
proveniencia_df.to_csv(proveniencia_file, index=False, encoding="utf-8")
print(f"Registro de proveniencia salvo em: {proveniencia_file}")
proveniencia_df


## 8. Tratamento dos dados

### 8.1 Tratamento IBGE

*Preencher posteriormente.*

### 8.2 Tratamento DIEESE

*Preencher posteriormente.*

### 8.3 Padronização

*Preencher posteriormente.*

In [ ]:
# 8. Tratamento e padronização dos dados
# Preencher posteriormente.

## 9. Integração das fontes

### 9.1 Chave de integração

Além da chave entre a Fonte 1 (IBGE/SIDRA) e a Fonte 2 (DIEESE), a Fonte 3 (TSE/Dados Abertos) será integrada por meio da **chave temporal `ano` (ano eleitoral)** e da **chave partidária `sigla_partido`**, permitindo cruzar a variação de preços e inflação com os períodos de governos, ciclos eleitorais e espectros partidários.

Resumo das três fontes e suas chaves de integração:

| Fonte | Método | Chave de integração principal |
|---|---|---|
| Fonte 1 — IBGE/SIDRA | API | `ano`/`mês`, `sigla_uf`/`municipio` |
| Fonte 2 — DIEESE | Scraping HTML | `cidade`/`sigla_uf`, `mês`/`ano` |
| Fonte 3 — TSE/Dados Abertos | API | `ano` (eleitoral), `sigla_partido` |

*Preencher posteriormente.*

### 9.2 Junção

*Preencher posteriormente.*

### 9.3 Verificação

*Preencher posteriormente.*

In [ ]:
# 9.4 Integração e verificação
# Preencher posteriormente.

## 10. Base final

### 10.1 Estrutura

*Preencher posteriormente.*

### 10.2 Quantidade de registros

*Preencher posteriormente.*

### 10.3 Exportação

*Preencher posteriormente.*

In [ ]:
# Exportação da base final
# Preencher posteriormente.

## 11. Proveniência e observações

*Preencher posteriormente.*